<a href="https://colab.research.google.com/github/Sky388-bit/ITCS-3162-Data-Mining/blob/main/Copy_of_lab_03_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Exercise: Wrangling the Google Play Store Dataset

**ITCS 3162 — Introduction to Data Mining**

**Name:** Andrew Laughlin
**Date:** 05/31/2026

This dataset is *messier* than diamonds — it's scraped from the Google Play Store and has the kinds of problems you'll see in the wild: numbers stored as strings, units mixed with values, duplicates, weird placeholder strings, and missing values. That's the point: you'll do real cleaning.

**Source:** Kaggle's "Google Play Store Apps" dataset (mirrored on GitHub for stable URL access).

When you're done, **Restart & Run All**, download as `.ipynb`, and submit via Canvas.


## Setup

Run this cell. It loads the dataset directly from a GitHub URL.


In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/sumitgirwal/google-play-store-data-analysis/master/googleplaystore.csv"
df = pd.read_csv(URL)
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Exercise 1 — Initial inspection (10 pts)

In the cell below, print **all three**:
1. The shape of the DataFrame (rows, columns)
2. The output of `df.info()`
3. The output of `df.describe(include="all")`

Then in the markdown cell, answer the questions.


In [60]:
# TODO: print shape, info(), describe(include='all')
print(df.shape)
print(df.info())
print(df.describe(include="all"))

(10841, 13)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB
None
           App Category       Rating Reviews                Size    Installs  \
count    10841    10841  9367.000000   10841               10841  

**Answer the following:**

1. How many rows and columns does the dataset have?
2. Look at the `Reviews`, `Size`, `Installs`, and `Price` columns. What dtype does pandas assign them? Why is that a problem for a numeric column like `Reviews`?
3. Which columns appear to have missing values?

YOUR ANSWERS:
1. The dataset has 10841 rows and 13 columns.
2. It assigns them an object data type which is a problem for a numeric column like reviews because you can't do comparisons.
3. Ratings, type and current version all seem to have NaN values.


## Exercise 2 — Missing values (10 pts)

In the cell below, compute (a) the count of missing values per column, sorted descending, and (b) the **percentage** of missing values per column rounded to 2 decimal places. Then drop any row that is missing the `Rating` column (since it's the most important missingness to handle) and store the result in `df_r`.


In [61]:
# TODO: counts of missing values per column (sorted descending)
missing_values = df.isna().sum()
missing_values = missing_values.sort_values(ascending=False)
print(missing_values)
# TODO: percentage missing per column (rounded to 2 decimal places)
percentage = df.isna().mean()
percentage = round((percentage * 100), 2)
print(percentage)
# TODO: create df_r by dropping rows where Rating is NaN; print its new shape
df_r = df.dropna(subset=["Rating"])
print(df_r.shape)

Rating            1474
Current Ver          8
Android Ver          3
Content Rating       1
Type                 1
Size                 0
Reviews              0
Category             0
App                  0
Price                0
Installs             0
Last Updated         0
Genres               0
dtype: int64
App                0.00
Category           0.00
Rating            13.60
Reviews            0.00
Size               0.00
Installs           0.00
Type               0.01
Price              0.00
Content Rating     0.01
Genres             0.00
Last Updated       0.00
Current Ver        0.07
Android Ver        0.03
dtype: float64
(9367, 13)


## Exercise 3 — Duplicates (10 pts)

There are duplicate rows in this dataset (the same app appears multiple times). In the cell below:
1. Print how many fully-duplicate rows exist in `df_r`.
2. How many duplicate values are there in the `App` column specifically (apps appearing under different categories)?
3. Create `df_dedup` by dropping fully-duplicate rows and print its new shape.


In [62]:
# TODO: fully duplicate rows
duplicates = df_r.duplicated()
print(duplicates)
# TODO: duplicate App names (use df_r["App"].duplicated().sum())
print(df_r["App"].duplicated().sum())

# TODO: df_dedup = df_r.drop_duplicates(); print shape
df_dedup = df_r.drop_duplicates()
print(df_dedup.shape)

0        False
1        False
2        False
3        False
4        False
         ...  
10834    False
10836    False
10837    False
10839    False
10840    False
Length: 9367, dtype: bool
1170
(8893, 13)


## Exercise 4 — Cleaning the `Installs` column (15 pts)

The `Installs` column looks like `"1,000,000+"` — a string with a `+` and commas. Convert it to an integer column. Steps:

1. Inspect a few unique values: `df_dedup["Installs"].unique()[:10]`
2. Remove `+` and `,` characters with `.str.replace()`
3. Convert to int with `.astype(int)` (or use `pd.to_numeric` with `errors="coerce"` if any values look weird)
4. Assign the cleaned values back to `df_dedup["Installs"]`
5. Verify with `df_dedup["Installs"].dtype` and `.head()`


In [63]:
print(df_dedup["Installs"].unique()[:10])
clean = df_dedup["Installs"].str.replace("+", "", regex=False)
clean = clean.str.replace(",", "", regex=False)

# TODO: clean Installs to integer
clean = pd.to_numeric(clean, errors="coerce")
df_dedup.loc[:, "Installs"] = clean
print(df_dedup["Installs"].dtype)
print(df_dedup["Installs"].head())

['10,000+' '500,000+' '5,000,000+' '50,000,000+' '100,000+' '50,000+'
 '1,000,000+' '10,000,000+' '5,000+' '100,000,000+']
object
0       10000.0
1      500000.0
2     5000000.0
3    50000000.0
4      100000.0
Name: Installs, dtype: object


## Exercise 5 — Cleaning the `Price` column (15 pts)

`Price` looks like `"0"` for free apps and `"$4.99"` for paid ones. Convert it to a float.

1. Use `.str.replace("$", "", regex=False)` then `.astype(float)`.
2. Watch for any unexpected values (the real dataset has at least one weird row — the column may have a non-numeric placeholder that `pd.to_numeric(..., errors="coerce")` handles gracefully).
3. After cleaning, print how many paid apps there are (where `Price > 0`).


In [64]:
# TODO: clean Price to float
clean_price = df_dedup["Price"].str.replace("$", "", regex=False)
clean_price = pd.to_numeric(clean_price, errors="coerce").astype(float)
df_dedup.loc[:, "Price"] = clean_price
# TODO: count paid apps
print(sum(df_dedup['Price'] > 0))


613


## Exercise 6 — Filtering and a derived column (15 pts)

1. Create `top_apps`, containing only apps with `Rating >= 4.5` and `Installs >= 1_000_000`.
2. Print its shape and the top 5 categories of those apps by count.
3. In `df_dedup`, add a new column `Revenue` defined as `Price * Installs` (zero for free apps).
4. Show the top 10 apps by `Revenue`.


In [65]:
# TODO: top_apps and category counts
top_apps = df_dedup[(df_dedup["Rating"] >= 4.5) & (df_dedup["Installs"] >= 1_000_000)]
print(top_apps.shape)
print(top_apps["Category"].value_counts())
df_dedup['Revenue'] = df_dedup['Price'] * df_dedup["Installs"]
# TODO: Revenue column and top 10
top_ten = df_dedup['Revenue'].sort_values(ascending=False)
print(top_ten[:10])

(1234, 13)
Category
GAME                   272
FAMILY                 171
TOOLS                   89
HEALTH_AND_FITNESS      82
PRODUCTIVITY            62
PHOTOGRAPHY             58
SHOPPING                47
EDUCATION               39
PERSONALIZATION         37
SOCIAL                  36
SPORTS                  33
FINANCE                 32
BOOKS_AND_REFERENCE     30
LIFESTYLE               27
COMMUNICATION           27
VIDEO_PLAYERS           25
NEWS_AND_MAGAZINES      22
TRAVEL_AND_LOCAL        21
FOOD_AND_DRINK          20
MEDICAL                 15
ENTERTAINMENT           13
MAPS_AND_NAVIGATION     13
HOUSE_AND_HOME          11
BUSINESS                11
WEATHER                 10
AUTO_AND_VEHICLES        7
COMICS                   6
ART_AND_DESIGN           4
BEAUTY                   4
PARENTING                4
DATING                   3
EVENTS                   2
LIBRARIES_AND_DEMO       1
Name: count, dtype: int64
4347    69900000.0
2241    69900000.0
5351    39999000.0
5356  

/tmp/ipykernel_12014/4226166817.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_dedup['Revenue'] = df_dedup['Price'] * df_dedup["Installs"]


## Exercise 7 — Group-by summary (15 pts)

Using `df_dedup`, produce a single summary table with one row per **Category**, containing:
- `n_apps` — number of apps in that category
- `mean_rating` — average rating (rounded to 2 decimals)
- `mean_installs` — average installs (rounded to 0 decimals)
- `total_revenue` — sum of `Revenue`

Sort by `total_revenue` descending. Show only the top 10 rows.


In [66]:
# TODO: groupby summary
summary = df_dedup.groupby("Category")
summary=  summary.agg(n_apps=("App", "count"),mean_rating=("Rating", lambda x: round(x.mean(), 2)),mean_installs=("Installs", lambda x: round(x.mean(), 0)),total_revenue=("Revenue", "sum"))
#I used AI on this to help me with the formating and what function to use
summary= summary.sort_values("total_revenue", ascending=False).head(10)


print(summary)


                 n_apps  mean_rating  mean_installs total_revenue
Category                                                         
FAMILY             1718         4.19      5844663.0   185774296.7
LIFESTYLE           305         4.10      1753250.0    57583939.4
GAME               1074         4.28     29370449.0   40986840.88
FINANCE             317         4.13      2430008.0    25726644.0
PHOTOGRAPHY         304         4.18     31977773.0     8941049.8
MEDICAL             302         4.18       139612.0     8371355.0
PERSONALIZATION     310         4.33      6691461.0     7786309.8
TOOLS               734         4.05     15600442.0     5462910.3
SPORTS              286         4.23      5344516.0     4706154.0
PRODUCTIVITY        334         4.20     37314581.0     4304451.9


## Exercise 8 — Reflection (10 pts)

In 4–6 sentences, answer all of:

1. Which cleaning step would have been impossible to skip if you wanted to compute average install counts? Why?
2. What's one piece of information in the raw data that you *could* extract with more work but didn't (e.g., the `Size` column, the `Last Updated` column)? How would you approach it?
3. Did anything in the data surprise you?

YOUR ANSWER:
1. The installation cleaning step would've been impossible if the + and , weren't removed from the data before hand. This wouldn't have worked because you cant take the average of a string.
2. The size one would be fairly easy to extract. ALl you would have to do is drop the non numerics from the string and use that.
3. The amount of weirds variations in installs.


## Submission checklist

- [ ] Name and date filled in
- [ ] All TODO cells completed and run
- [ ] All `YOUR ANSWER` prompts replaced
- [ ] **Restart & Run All** completes without errors
- [ ] Downloaded as `.ipynb` and uploaded to Canvas
